# 232 — High-Gamma (HG) clustering

Loads ERSPs from `01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY/<pid>/LM/ERSP_matrix/<cond>/*.npy`, collapses the freq axis to a single high-gamma band time series (70–150 Hz by default), and clusters on the resulting (n_samples × n_time) feature matrix.

Outputs land in `outputs/clustering/{kmeans,hierarchical}/hg/runs/<timestamp>/` via `lf_cluster_run.fit_and_save`.

Per-sample HG sparkline PNGs go under `04_ersp_LM_RAWONLY/<pid>/LM/ERSP_hg/<cond>/<stem>_HG.png` so the MOBA samples-pane toggle can switch to the HG view.


## Config

## Load canonical dataset (shared across 210/230/231/232)

In [4]:
# ── Canonical dataset (shared by 210/230/231/232) ──────────────────
# Loads ERSPs from INPUT_DIR, drops non-neural channels, then gates by
# high-activity. SAME filter for every clustering notebook so cross-
# feature-set / cross-method comparisons are on the IDENTICAL sample set.
# Cached in 02_FBM_Clustering/outputs/_dataset/canonical/ — subsequent
# notebook runs load instantly instead of re-walking the ERSP_matrix tree.
from pathlib import Path
from functions.lf_dataset import prepare_dataset, DEFAULT_CACHE_DIR

INPUT_DIR = Path(r'\\\\nasac-m2.unige.ch\\m-HumanNeuronLab\\ANALYSIS\\FLM\\Analysis_LoraFanda\\01_FBM_Analysis\\outputs\\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = Path('../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY').resolve()

df_meta, ersp_list, X_3d = prepare_dataset(INPUT_DIR, cache_dir=DEFAULT_CACHE_DIR)
print(f'\nCanonical dataset: {len(df_meta)} samples · X_3d.shape={X_3d.shape}')


[lf_dataset cache hit] \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\_dataset\canonical
  1538 samples · X_3d.shape=(1538, 129, 300)

Canonical dataset: 1538 samples · X_3d.shape=(1538, 129, 300)


## 1 — Load ERSPs (same source as 210/230)

In [6]:
from functions.lf_clustering import s10_load_ersps
df_meta, ersp_list = s10_load_ersps(
    input_dir=INPUT_DIR,
    task=TASK,
    allowed_conditions=CONDITIONS,
    n_freq=N_FREQ,
    n_time=N_TIME,
)
print(df_meta.shape, len(ersp_list))
df_meta.head()


NameError: name 'TASK' is not defined

## 2 — Build HG feature matrix

In [3]:
X_hg = build_hg_feature_matrix(ersp_list, hg_band=HG_BAND, fmax=FMAX)
print('X_hg:', X_hg.shape, '  dtype:', X_hg.dtype)


[build_hg_feature_matrix] X_hg.shape=(7268, 300)  hg_band=(70.0, 150.0) Hz  fmax=500.0 Hz
X_hg: (7268, 300)   dtype: float32


## 3 — Per-sample HG sparkline PNGs

Writes one PNG per sample under `01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY/<pid>/LM/ERSP_hg/<cond>/<stem>_HG.png`.
MOBA's samples-pane toggle (`ERSP | Blob | -101 | HG`) reads from here.
Idempotent — skip-if-exists unless `FORCE_REGEN = True`.


In [4]:
GEN_HG_PNGS = True
FORCE_REGEN = False

def _hg_view_path(file_path):
    p = str(file_path).replace('\\\\', '/').replace('\\', '/')
    if '/ERSP_matrix/' not in p:
        return None
    p = p.replace('/ERSP_matrix/', '/ERSP_hg/').replace('.npy', '_HG.png')
    return Path(p)

if not GEN_HG_PNGS:
    print('[skip] GEN_HG_PNGS = False')
else:
    n_total = len(ersp_list)
    n_wrote = n_skip = n_bad = 0
    print(f'Writing per-sample HG sparkline PNGs for {n_total} samples...')
    for i in range(n_total):
        fp = df_meta.iloc[i].get('file_path')
        if not fp:
            n_bad += 1; continue
        out_path = _hg_view_path(fp)
        if out_path is None:
            n_bad += 1; continue
        try:
            if FORCE_REGEN or not out_path.exists():
                save_sample_hg_png(ersp_list[i], out_path, hg_band=HG_BAND, fmax=FMAX)
                n_wrote += 1
            else:
                n_skip += 1
        except Exception as e:
            print(f'  [warn] sample {i}: {e}')
            n_bad += 1
        if (i + 1) % 500 == 0:
            print(f'  ...{i+1}/{n_total}')
    print(f'HG PNGs : wrote {n_wrote}, skipped existing {n_skip}')
    if n_bad:
        print(f'  ({n_bad} samples without parseable file_path — skipped)')


Writing per-sample HG sparkline PNGs for 7268 samples...
  ...500/7268
  ...1000/7268
  ...1500/7268
  ...2000/7268
  ...2500/7268
  ...3000/7268
  ...3500/7268
  ...4000/7268
  ...4500/7268
  ...5000/7268
  ...5500/7268
  ...6000/7268
  ...6500/7268
  ...7000/7268
HG PNGs : wrote 0, skipped existing 7268


# Clustering

Two methods on the same `X_hg` (HG time series, shape n_samples × 300): K-Means K-sweep + Hierarchical K-sweep.
Each `fit_and_save` call writes a self-contained run directory and updates `outputs/clustering/index.json`.


In [5]:
# KMeans K-sweep on HG features.
manifest_km = R.fit_and_save(
    X_hg,
    df_keep=df_meta,
    method='kmeans',
    feature_set='hg',
    params={'k_range': KMEANS_K_RANGE, 'random_state': RANDOM_STATE, 'n_init': 20},
    method_label='K-Means',
    feature_set_label='High-Gamma Time Series',
    notebook=SCRIPT_NAME,
)
BEST_K = manifest_km['summary']['best_k']
print(f'Best K (KMeans/hg, by silhouette): {BEST_K}')


  K= 10  sil=0.0892
  K= 11  sil=0.0779
  K= 12  sil=0.0766
  K= 13  sil=0.0768
  K= 14  sil=0.0697
  K= 15  sil=0.0693
  K= 16  sil=0.0657
  K= 17  sil=0.0611
  K= 18  sil=0.0614
  K= 19  sil=0.0621
  K= 20  sil=0.0579

[KMeans] Best K=10  silhouette=0.0892
[fit_and_save] kmeans/hg/20260522_205451
  n_samples=7268 n_clusters=10 silhouette=0.089
  -> \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering\kmeans\hg\runs\20260522_205451
Best K (KMeans/hg, by silhouette): 10


In [6]:
# Hierarchical K-sweep on HG features.
manifest_hc = R.fit_and_save(
    X_hg,
    df_keep=df_meta,
    method='hierarchical',
    feature_set='hg',
    params={'linkage': HC_METHOD, 'metric': HC_METRIC, 'k_range': KMEANS_K_RANGE},
    method_label='Hierarchical',
    feature_set_label='High-Gamma Time Series',
    notebook=SCRIPT_NAME,
)
print(f'Best K (HC/hg, by silhouette): {manifest_hc["summary"]["best_k"]}')


[HC] method=ward  metric=euclidean  n=7268  cophenetic_r=0.361
  HC sweep k=10: silhouette=0.053
  HC sweep k=11: silhouette=0.056
  HC sweep k=12: silhouette=0.059
  HC sweep k=13: silhouette=0.052
  HC sweep k=14: silhouette=0.029
  HC sweep k=15: silhouette=0.030
  HC sweep k=16: silhouette=0.030
  HC sweep k=17: silhouette=0.031
  HC sweep k=18: silhouette=0.032
  HC sweep k=19: silhouette=0.033
  HC sweep k=20: silhouette=0.034
[fit_and_save] hierarchical/hg/20260522_205603
  n_samples=7268 n_clusters=12 silhouette=0.059
  -> \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering\hierarchical\hg\runs\20260522_205603
Best K (HC/hg, by silhouette): 12


## Per-cluster centroid sparklines (for the MOBA cluster chips)

Walks `index.json`, for every `feature_set == 'hg'` run, plots the mean HG time series per cluster as a sparkline and writes `<run_dir>/cluster_centroids/cluster_<NN>.png`. MOBA picks them up automatically.


In [7]:
# BACKFILL_CENTROIDS for HG: mean-HG sparkline per cluster.
import json

CLUSTERING_DIR = Path(R.DEFAULT_OUTPUTS_ROOT)
INDEX_PATH = CLUSTERING_DIR / 'index.json'

def _save_per_cluster_centroid_pngs_hg(manifest, X_local):
    if manifest['feature_set'] != 'hg':
        return 0
    run_dir = CLUSTERING_DIR / manifest['method'] / manifest['feature_set'] / 'runs' / manifest['run_id']
    df = pd.read_csv(run_dir / 'labels.csv')
    cluster_col = f"cluster_{manifest['method']}_{manifest['feature_set']}"
    if cluster_col not in df.columns:
        cands = [c for c in df.columns if c.startswith('cluster_')]
        if not cands: return 0
        cluster_col = cands[0]
    labels = df[cluster_col].to_numpy()
    if len(labels) != X_local.shape[0]:
        print(f"  [skip] {manifest['run_id']}: labels ({len(labels)}) vs X_hg ({X_local.shape[0]}) mismatch")
        return 0
    out_dir = run_dir / 'cluster_centroids'
    out_dir.mkdir(parents=True, exist_ok=True)
    uniq = sorted(int(c) for c in np.unique(labels))
    for c in uniq:
        idx = np.where(labels == c)[0]
        mean_hg = X_local[idx].mean(axis=0)
        fig, ax = plt.subplots(figsize=(2.4, 1.7))
        render_hg_sparkline(ax, mean_hg, ylim=(-6, 6))
        fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
        out_path = out_dir / f'cluster_{int(c):02d}.png'
        fig.savefig(out_path, dpi=90, bbox_inches='tight', pad_inches=0)
        plt.close(fig)
    return len(uniq)

if not INDEX_PATH.exists():
    print(f'No {INDEX_PATH} yet — run the fit_and_save cells above first.')
else:
    with open(INDEX_PATH) as f:
        idx_data = json.load(f)
    runs = [r for r in idx_data.get('runs', []) if r['feature_set'] == 'hg']
    print(f'Backfilling per-cluster centroid PNGs for {len(runs)} hg runs...')
    for run in runs:
        manifest_path = CLUSTERING_DIR / run['path'] / 'manifest.json'
        if not manifest_path.exists():
            print(f"  [skip] {run['path']}: missing manifest.json"); continue
        with open(manifest_path) as f:
            manifest = json.load(f)
        n = _save_per_cluster_centroid_pngs_hg(manifest, X_hg)
        if n:
            print(f"  [{manifest['method']}/{manifest['feature_set']}] {manifest['run_id']}  ->  {n} cluster PNGs")
    print('\nDone. Commit and push.')


Backfilling per-cluster centroid PNGs for 4 hg runs...
  [hierarchical/hg] 20260521_160447  ->  12 cluster PNGs
  [hierarchical/hg] 20260522_205603  ->  12 cluster PNGs
  [kmeans/hg] 20260521_160345  ->  10 cluster PNGs
  [kmeans/hg] 20260522_205451  ->  10 cluster PNGs

Done. Commit and push.
